# 02 - Generate reasoning chains
For every MATH training problem (MATH-500 problems removed), the base model writes 4 attempts, and we check each with math_verify.
Run on Kaggle (2x T4). To finish faster, the problems were split in two halves and each half ran on its own GPU at the same time; the code per half is exactly this notebook. Output: `datasets/chains_qwen2b.json`.

In [ ]:
!pip uninstall -y torchaudio # newer Kaggle image: old torchaudio breaks "import datasets" after vllm upgrades torch
!pip install -U transformers vllm math-verify datasets

In [ ]:
import sys
import torch
import transformers
import vllm
import datasets

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("vLLM:", vllm.__version__)
print("Datasets:", datasets.__version__)
print("GPUs:", torch.cuda.device_count())

In [ ]:
from datasets import load_dataset

math_train=load_dataset("DigitalLearningGmbH/MATH-lighteval","default",split="train")
math500=load_dataset("HuggingFaceH4/MATH-500",split="test")

print(math_train)
print(math500)

In [ ]:
def extract_boxed(text: str):
    start=text.rfind('\\boxed{')
    if start==-1:
        return None

    i=start+len('\\boxed{')
    depth=1
    while i<len(text) and depth>0:
        if text[i]=='{':
            depth+=1
        elif text[i]=='}':
            depth-=1
        i+=1

    if depth!=0:
        return None
    return text[start+len('\\boxed{'):i-1]

In [ ]:
test_questions=set(i['problem'].strip() for i in math500)

problems=[]

for i in math_train.shuffle(seed=0):
    gold=extract_boxed(i['solution'])
    if gold is None:
        continue
    if i['problem'].strip() in test_questions:
        continue
    problems.append({
        'id': 'math-'+str(len(problems)),
        'level': i['level'],
        'type': i['type'],
        'problem': i['problem'],
        'gold': gold,
        'ref_solution': i['solution'],
    })

print(len(problems))
problems[0]

In [ ]:
shard=0    # the run can be split in halves that run on Kaggle at the same time (shard 0 and 1)
n_shards=1 # 1 = everything in one run

problems=problems[shard::n_shards]
print(len(problems))

In [ ]:
def format_prompt(record: dict)->str:
    result=""

    result+='Question: '+record.get('problem')+'\n'

    result+='Please reason step by step, and put your final answer within \\boxed{}.'+'\n'

    result+='Answer:'

    return result

In [ ]:
prompts=[format_prompt(i) for i in problems]
print(prompts[0])

In [ ]:
from vllm import LLM, SamplingParams

model_name='Qwen/Qwen3.5-2B-Base' # 4B run: 'Qwen/Qwen3.5-4B-Base'
model_tag='qwen2b'                 # 4B run: 'qwen4b'
tensor_parallel_size=1             # 2 = split the model over both T4 GPUs (faster for 4B)

llm=LLM(
    model=model_name,
    tensor_parallel_size=tensor_parallel_size,
    dtype="float16",
    max_model_len=2048,
    gpu_memory_utilization=0.88,
    limit_mm_per_prompt={"image": 0, "video": 0},
)

print("Model loaded")

In [ ]:
sample_params=SamplingParams(
    n=4,
    temperature=0.8,
    top_p=0.95,
    max_tokens=1024,
    stop=["\n\nQuestion:", "Question:", "<|endoftext|>", "<|im_end|>"],
    seed=0,
)

In [ ]:
outputs=llm.generate(prompts,sample_params)
print("Generation done")

In [ ]:
from math_verify import parse, verify

def is_correct(text: str, gold: str)->bool:
    try:
        return bool(verify(parse('$'+gold+'$'),parse(text)))
    except Exception:
        return False

In [ ]:
for record,output in zip(problems,outputs):
    chains=[]
    for j in output.outputs:
        pred=extract_boxed(j.text)
        chains.append({
            'text': j.text,
            'pred': pred,
            'ok': is_correct(j.text,record['gold']) if pred else False,
            'finished': j.finish_reason=='stop' and pred is not None,
            'n_tokens': len(j.token_ids),
        })
    record['chains']=chains

problems[0]['chains'][0]

In [ ]:
n=len(problems)
correct_counts=[sum(c['ok'] for c in r['chains']) for r in problems]

print("Chain accuracy:", sum(correct_counts)/(n*4))
print("All wrong:", sum(c==0 for c in correct_counts)/n)
print("All right:", sum(c==4 for c in correct_counts)/n)
print("Mixed:", sum(0<c<4 for c in correct_counts)/n)
print("First wrong but some right:", sum((not r['chains'][0]['ok']) and c>0 for r,c in zip(problems,correct_counts))/n)

In [ ]:
import json
from pathlib import Path

savepath=Path(f'/kaggle/working/chains_{model_tag}_shard{shard}.json')

with savepath.open('w',encoding='utf-8') as file:
    json.dump(problems,file,ensure_ascii=False,indent=2)

print("Saved:",savepath)